first import necessary modules

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
from src.cleaning_and_helpers import clean_rbcl_data

c:\Users\rah10\miniconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
pd.set_option('display.max_columns', None)

Import and clean data frames from all projects

In [4]:
# sky islands data
# TODO update filepath when sky islands rbcl data is ready
df_si = clean_rbcl_data('data/SIpollen.csv',
                          '../pollenGeolocation_saved/SIbeeRBCL.csv')
# ffar data
# TODO update filepath when ffar rbcl data is ready
df_ffar = clean_rbcl_data('data/FFARpollen.csv',
                          '../pollenGeolocation_saved/FFARbeeRBCL.csv')

# ncasi data
# TODO update filepath when ffar rbcl data is ready
df_ncasi = clean_rbcl_data('data/NCASIpollen.csv',
                          '../pollenGeolocation_saved/NCASIbeeRBCL.csv',
                          format="ISO-8859-1")


now we will set up a df where we can join these datasets using concat

In [12]:
df_all = pd.concat([df_si,df_ffar,df_ncasi], axis=0, ignore_index=True)

# this introduces NAs again where the RBCL sequences of the SI dataset were not detected in the FFAR
# data and vice versa; use .fillna(0) to change those to 0s
df_all = df_all.fillna(0)
df_all.to_csv('../pollenGeolocation_saved/ALLbeeRBCL.csv')

In [13]:
# saving out lat and long coords
df_all['lat_long'] = [', '.join(str(x) for x in y) for y in map(tuple, df_all[['Lat', 'Long']].values)]
np.save('../pollenGeolocation_saved/latlong.npy', df_all['lat_long'])

In [14]:
rbcl_cols = [col for col in df_all if col.startswith('RBCL')]
my_cols = ['Lat', 'Long', 'lat_long'] + rbcl_cols
df_all = df_all[my_cols]
df_all = df_all.reindex(columns=my_cols)

In [15]:
# saving out features for modeling
X = df_all.drop('Lat',axis=1)
X = X.drop('Long',axis=1)
X = X.drop('lat_long',axis=1)
X.columns
np.save('../pollenGeolocation_saved/X_cols.npy', X.columns)
np.save('../pollenGeolocation_saved/X.npy', X)

In [16]:
# saving out targets for modeling
lat = np.array(df_all['Lat'])
long = np.array(df_all['Long'])
y = np.column_stack((lat, long))
np.save('../pollenGeolocation_saved/y.npy', y)